In [ ]:
# Full name
NAME = ""
# Institutional email (hm.edu or hmtm.de)
EMAIL = ""

<a href="https://colab.research.google.com/github/aica-wavelab/aica-assignments/blob/main/A2_generation/9_8_ml_transformer.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Melody Generation with Machine Learning

+ **AI in Culture and Arts - Tech Crash Course**
+ **Date:** 21.05.2026
+ **Author:** Dr. Benedikt Zönnchen

In the following we will create music sheets and sound. For those tasks ``Python`` requires external programs that you should install if you are working locally:

1. [Musescore](https://musescore.org/de) (for generating sheets)
2. [FluidSynth](https://www.fluidsynth.org/) (for generating sound)

If you are working on google ``Colab``, you can evaluate the following two cells to install these applications:

In [ ]:
#@title install dependencies to play sound
%%capture
print('installing fluidsynth...')
!apt-get install fluidsynth > /dev/null
!cp /usr/share/sounds/sf2/FluidR3_GM.sf2 ./font.sf2
print('done!')

In [ ]:
#@title install dependencies to show score in music notation
%%capture
print('installing musescore3...')
!apt-get install musescore3 > /dev/null
print('done!')

Furthermore, for this notebook we need the following ``Python`` packages and modules. Execute the cell to install them:

In [ ]:
#@title Setup: install required Python packages

%pip install music21
%pip install pyfluidsynth

%pip install matplotlib
%pip install seaborn

%pip install pandas
%pip install numpy
%pip install torch
%pip install torchview

%pip install otter-grader==5.5.0


In [ ]:
#@title Setup: download assignment files (run this cell)

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # download test files
    import requests, os

    folders = ['tests', 'data', 'figs']
    link = "https://api.github.com/repos/aica-wavelab/aica-assignments/contents/A2_generation"

    def download(entry, dest):
        if entry.get('type') != 'file' or not entry.get('download_url'):
            return
        r = requests.get(entry['download_url'])
        r.raise_for_status()
        with open(dest, 'wb') as out:
            out.write(r.content)

    for folder in folders:
        os.makedirs(folder, exist_ok=True)
        for f in requests.get(f"{link}/{folder}").json():
            download(f, f"{folder}/{f['name']}")

    for f in requests.get(link).json():
        if f['name'].endswith('.py'):
            download(f, f['name'])

    # Initialize Otter
    import otter
    grader = otter.Notebook(colab=True)
else:
    import otter
    grader = otter.Notebook('9_8_ml_transformer.ipynb')

## 25 Decoder-Only Transformer

<img src="figs/decoder.png" alt="" height="600">

### Mathematical Assumptions: Autoregression

Where in [23 Learning the Markov Matrix](https://colab.research.google.com/github/aica-wavelab/aica-assignments/blob/main/A2_generation/9_6_ml_ffn_markov.ipynb) we assumed the Markov condition, that is, that any the probability of the next event occuring only depends on the last event, we know relax this strong and rather unreasonable assumption.

Instead we assume a sequence of random variables $X_1, X_2, \ldots, X_k$ that are autoregressively dependent, that is,

\begin{align*}
P(x_{k+1}, x_k, x_{k-1}, \ldots, x_{1}) = P(x_{k+1} | x_{k}, \ldots, x_{1}) \cdot P(x_{k} | x_{k-1}, \ldots, x_{1}) \cdot \cdots \cdot P(x_2 | x_1) \cdot P(x_1)
\end{align*}

Here we wrote $P(x_1)$ as a shorthand for $P(X_1 = x_1)$. In other words we assume that the next musical event / note depends on all the events happened / computed before!

---

🗣 **Hint:** All large language model work under this assumption. They are autoregressive models!

---

### From Recurrence to Attention

In the previous notebook ([24 Recurrent Neural Networks (RNN) and LSTM](https://colab.research.google.com/github/aica-wavelab/aica-assignments/blob/main/A2_generation/9_7_ml_rnn.ipynb)) we used an **LSTM** to generate melodies. The LSTM works by processing one token at a time, maintaining a **hidden state** that summarises everything seen so far. While LSTMs are powerful, they have a fundamental limitation: information has to travel through the hidden state, which can get "washed out" over long sequences — even with the gating mechanisms of an LSTM.

In this notebook we explore the **transformer** architecture, specifically a *decoder-only* transformer similar to GPT. Transformers were introduced in 2017 in the landmark paper *[Attention Is All You Need](https://arxiv.org/abs/1706.03762)* and have since become the dominant architecture for text generation, language translation, and increasingly also for symbolic music generation.

The central innovation of the transformer is the **self-attention mechanism**, which allows every token in the sequence to directly attend to (i.e. be influenced by) every other token — without having to route information through a sequential hidden state. This has two important consequences:

1. **Long-range dependencies** are handled more directly: a note near the end of a melody can directly attend to a note near the beginning.
2. **Parallelism**: all positions are processed simultaneously, which makes transformers much faster to train on GPUs than RNNs.

The price to pay is **memory**: the attention mechanism requires storing an $n \times n$ matrix for a sequence of length $n$, so very long sequences become expensive.

We will build the transformer bottom-up:
1. A single **attention head** — the core building block
2. **Multi-head attention** — multiple heads running in parallel
3. A **transformer block** — attention + feed-forward network + layer normalisation
4. The full **decoder-only transformer** — a stack of blocks

### The Self-Attention Mechanism

The key idea of self-attention is **routing**: every output position is computed as a *weighted sum* of all input positions, where the weights (the *attention scores*) express how relevant each input position is for the current output position.

Concretely, given a sequence of $T$ token embeddings $\mathbf{x}_0, \ldots, \mathbf{x}_{T-1} \in \mathbb{R}^{d}$, a single attention head computes three sets of vectors:

$$\mathbf{q}_j = \mathbf{W}_q\, \mathbf{x}_j \quad \text{(query — what am I looking for?)}$$

$$\mathbf{k}_i = \mathbf{W}_k\, \mathbf{x}_i \quad \text{(key — what do I offer?)}$$

$$\mathbf{v}_i = \mathbf{W}_v\, \mathbf{x}_i \quad \text{(value — what information do I carry?)}$$

The weight matrices $\mathbf{W}_q, \mathbf{W}_k, \mathbf{W}_v \in \mathbb{R}^{D_q \times d}$ are **learned** (here $D_q$ is called the `head_size`). The **attention score** between position $j$ (the query) and position $i$ (the key) measures how similar the query and the key are:

$$e_{j,i} = \frac{\mathbf{k}_i^\top \mathbf{q}_j}{\sqrt{D_q}}$$

The division by $\sqrt{D_q}$ prevents the dot products from becoming very large (which would push the softmax into regions of near-zero gradient). Normalising with softmax gives attention weights that sum to one:

$$\alpha({\mathbf{x}_j,\mathbf{x}_i}) = \frac{\exp(e_{j,i})}{\sum_{r=0}^{T-1} \exp(e_{j,r})}$$

Finally, the output at position $j$ is a weighted sum of the values:

$$\text{head}_j = \sum_{i=0}^{T-1} \alpha({\mathbf{x}_j,\mathbf{x}_i})\, \mathbf{v}_i$$

<img src="figs/routing.png" alt="" height="500">

In matrix form (stacking all queries, keys and values into matrices $\mathbf{Q}, \mathbf{K}, \mathbf{V}$):

$$\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{Softmax}\!\left(\frac{\mathbf{K}^\top \mathbf{Q}}{\sqrt{D_q}}\right)\mathbf{V}$$

<img src="figs/self-attention.png" alt="" height="500">

#### Causal Masking

Our transformer is used for *generation*: given notes $0, 1, \ldots, t-1$, predict note $t$. At generation time, the future notes do not exist yet. To make training consistent with generation, we **mask** future positions during training by replacing $e_{j,i}$ with $-\infty$ for all $i > j$. After softmax, $-\infty$ becomes $0$ weight, so the model effectively ignores future positions.

The mask has the shape of a lower-triangular matrix:

```
position:  0  1  2  3
query 0: [ 1  0  0  0 ]   position 0 can only see itself
query 1: [ 1  1  0  0 ]   position 1 can see 0 and 1
query 2: [ 1  1  1  0 ]   position 2 can see 0, 1, and 2
query 3: [ 1  1  1  1 ]   position 3 can see everything
```

The following video display beautifully the attention given to past events to produce the next one. This was produced by a special transformer called [Music Transformer](https://arxiv.org/abs/1809.04281) that uses a *relative postional encoding* which we do not use here. Furthermore, our model will only produce simple monophonic melodies.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import music21 as m21

import zipfile
from files import load_midi_files
from pianoroll import stream_to_df, plot_df
from encoder import PianoRollEncoder, StringToIntEncoder, TERM_SYMBOL

In [ ]:
from IPython.display import Video
Video("https://magenta.withgoogle.com/assets/music_transformer/motifs_visualization.mp4")

### Transformer Head

<img src="figs/transformer-head.png" alt="" height="400">

In [ ]:
class Head(nn.Module):
    """One masked self-attention head."""

    def __init__(self, n_embd, head_size, sequence_len, dropout):
        super().__init__()
        # Three learned linear projections — no bias needed
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # Lower-triangular mask stored as a non-trainable buffer
        self.register_buffer('tril', torch.tril(torch.ones(sequence_len, sequence_len)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape          # batch, time (sequence length), embedding dim

        k = self.key(x)            # (B, T, head_size)
        q = self.query(x)          # (B, T, head_size)
        v = self.value(x)          # (B, T, head_size)
        head_size = q.shape[-1]

        # Raw attention scores: (B, T, head_size) @ (B, head_size, T) -> (B, T, T)
        wei = q @ k.transpose(-2, -1) * (head_size ** -0.5)

        # Apply causal mask: set future positions to -inf so softmax gives them weight 0
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))

        # Normalise to get attention weights that sum to 1 across the key dimension
        wei = F.softmax(wei, dim=-1)   # (B, T, T)
        wei = self.dropout(wei)

        # Weighted sum of values: (B, T, T) @ (B, T, head_size) -> (B, T, head_size)
        out = wei @ v
        return out

### Multi-Head Attention

<img src="figs/multi-head.png" alt="" height="400">

Using a single attention head forces the model to learn one single "way of relating" tokens to each other. In practice it is much better to run **several heads in parallel**, each with its own $\mathbf{W}_q, \mathbf{W}_k, \mathbf{W}_v$ matrices. The hope is that different heads specialise in different kinds of relationships: one head might attend to rhythmic repetitions, another to harmonic context, and so on.

Each head works on a smaller, `head_size`-dimensional projection of the input (`head_size = n_embd // n_heads`). The outputs of all heads are concatenated and projected back to the original `n_embd` dimension via a learned matrix $\mathbf{W}_0$:

$$\text{MultiHead}(\mathbf{X}) = \left[\text{head}_1; \ldots; \text{head}_{n_h}\right] \mathbf{W}_0$$

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multiple attention heads running in parallel."""

    def __init__(self, n_embd, n_heads, head_size, sequence_len, dropout):
        super().__init__()
        self.heads = nn.ModuleList(
            [Head(n_embd, head_size, sequence_len, dropout) for _ in range(n_heads)]
        )
        # Project concatenated head outputs back to n_embd
        self.W0 = nn.Linear(head_size * n_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Run all heads and concatenate along the last dimension
        out = torch.cat([head(x) for head in self.heads], dim=-1)  # (B, T, head_size * n_heads)
        out = self.W0(out)          # (B, T, n_embd)
        out = self.dropout(out)
        return out

### Transformer Block

A **transformer block** wraps multi-head attention with two important additions:

**Feed-forward network (FFN)**: A small two-layer MLP applied *independently* to each position. This introduces non-linearity that the purely linear attention cannot provide. The hidden dimension is typically $4 \times$ the embedding dimension.

**Residual connections**: Instead of replacing the input, each sub-layer *adds* its output to the input: `x = x + sublayer(x)`. This is called a *residual* or *skip* connection. It ensures that gradients flow cleanly back to early layers during training, preventing vanishing gradients.

**Layer normalisation**: Applied *before* each sub-layer (this is the "Pre-LN" variant). It normalises the values across the embedding dimension so they stay in a well-behaved range throughout training.

Together, a block looks like:

<img src="figs/block.png" alt="" height="500">

In [ ]:
class FeedForward(nn.Module):
    """Position-wise feed-forward network."""

    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """One transformer block: (masked) multi-head attention + feed-forward."""

    def __init__(self, n_embd, n_heads, sequence_len, dropout):
        super().__init__()
        head_size = n_embd // n_heads          # split embedding evenly across heads
        self.sa   = MultiHeadAttention(n_embd, n_heads, head_size, sequence_len, dropout)
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1  = nn.LayerNorm(n_embd)
        self.ln2  = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))    # attention sub-layer with residual connection
        x = x + self.ffwd(self.ln2(x))  # feed-forward sub-layer with residual connection
        return x

### Positional Encoding and the Full Decoder

Unlike the LSTM, which processes tokens one after another and therefore *implicitly* knows their order, the transformer processes all positions **in parallel**. This means it has no built-in notion of position: without extra information, the sequence `C4 G4 E4` would look the same as `G4 E4 C4`.

We fix this by adding a **positional embedding** to each token embedding. For position $t$ in the sequence, we look up a learned vector $\mathbf{p}_t$ and add it to the token embedding $\mathbf{e}_t$:

$$\mathbf{x}_t = \mathbf{e}_t + \mathbf{p}_t$$

Both $\mathbf{e}_t$ and $\mathbf{p}_t$ have the same dimension `n_embd`. The positional embeddings are learned jointly with all other parameters during training.

<img src="figs/decoder.png" alt="" height="600">

The full model stacks $n_{\text{blocks}}$ transformer blocks and finishes with a linear layer that maps the `n_embd`-dimensional output at each position to a probability distribution over the vocabulary:

In [ ]:
class TransformerDecoder(nn.Module):
    """Decoder-only transformer for next-token prediction."""

    def __init__(self, vocab_size, sequence_len, n_embd, n_heads, n_blocks, dropout):
        super().__init__()
        # Learned token embedding: maps each integer token id to an n_embd-dimensional vector
        self.token_embedding_table    = nn.Embedding(vocab_size, n_embd)
        # Learned positional embedding: maps each position 0..sequence_len-1 to a vector
        self.position_embedding_table = nn.Embedding(sequence_len, n_embd)

        self.blocks = nn.Sequential(
            *[Block(n_embd, n_heads, sequence_len, dropout) for _ in range(n_blocks)]
        )
        self.ln_f   = nn.LayerNorm(n_embd)           # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size) # output projection

    def forward(self, idx):
        B, T = idx.shape

        # Token embeddings: each integer -> n_embd-dim vector
        token_emb = self.token_embedding_table(idx)                           # (B, T, n_embd)

        # Positional embeddings: position 0, 1, ..., T-1 -> n_embd-dim vectors
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))  # (T, n_embd)

        # Add token and positional embeddings (broadcasting over the batch dimension)
        x = token_emb + pos_emb                      # (B, T, n_embd)

        x = self.blocks(x)                           # (B, T, n_embd)
        x = self.ln_f(x)                             # (B, T, n_embd)
        logits = self.lm_head(x)                     # (B, T, vocab_size)
        return logits

### How the Transformer Trains Differently from the LSTM

There is an important difference in how the transformer is trained compared to the LSTM:

- **LSTM**: the model reads a window of `T` tokens and produces **one** prediction for the next token after the window.
- **Transformer**: the model reads a window of `T` tokens and produces **T predictions** simultaneously — one predicted next-token for *each* position in the window.

This means that from a single window of `T` tokens we get `T` training signals instead of just one. The target sequence is the input sequence **shifted by one position**:

```
Input:   [ C4,  G4,  E4,  F4,  G4 ]
Target:  [ G4,  E4,  F4,  G4,  A4 ]
          ▲                        
          at position 0 the model should predict G4
```

The causal mask ensures that when predicting the token at position $t$, the model can only see positions $0, \ldots, t-1$ — exactly as in generation.

### 25.1 Data Collection / Preparation

We use exactly the same dataset as in the LSTM notebook: ~100 German folk song melodies from ``data/deu_folk_songs.zip``, encoded in our piano-roll representation and transposed to C major.

In [ ]:
with zipfile.ZipFile('data/deu_folk_songs.zip', 'r') as zip_ref:
    zip_ref.extractall('data/deu_folk_songs/')

In [ ]:
# this can take some time!
import glob
time_step = 0.5
mid_files = glob.glob('data/deu_folk_songs/**/*.mid', recursive=True)
streams_folk_songs = load_midi_files(mid_files, time_step=time_step, transpose_to_major=True, max_files=100)
print(f'loaded {len(streams_folk_songs)} files')

In [ ]:
piano_roll_encoder = PianoRollEncoder(time_step=time_step)
piano_rolls, _ = piano_roll_encoder.encode_streams(streams_folk_songs)

string_to_int = StringToIntEncoder(piano_rolls)
piano_rolls_int = string_to_int.encode_sequences(piano_rolls)
vocab_size = len(string_to_int)

print(f'vocabulary size: {vocab_size}')
print(f'example piano roll (first 20 tokens): {piano_rolls_int[0][:20]}')

We now create overlapping windows. The **input** is a window of `sequence_len` tokens; the **target** is the same window **shifted by one position** (i.e. the next token at each position):

In [ ]:
sequence_len = 64   # hyper-parameter: how many past tokens the model can attend to

xs = []
ys = []
i_term_symbol = string_to_int.encode(TERM_SYMBOL)

for piano_roll in piano_rolls_int:
    # Pad the beginning so the model can also train on the start of each melody
    padded = [i_term_symbol] * sequence_len + piano_roll + [i_term_symbol]
    for i in range(len(padded) - sequence_len):
        xs.append(padded[i:(i + sequence_len)])
        ys.append(padded[(i + 1):(i + sequence_len + 1)])  # shifted by one!

print(f'number of training windows: {len(xs)}')
print(f'input  window (first 8 tokens): {xs[1][-8:]}')
print(f'target window (first 8 tokens): {ys[1][-8:]}')

In [ ]:
from torch.utils.data import TensorDataset, DataLoader, random_split

X = torch.tensor(xs, dtype=torch.long)   # (num_samples, sequence_len)
y = torch.tensor(ys, dtype=torch.long)   # (num_samples, sequence_len)  <-- note: 2-D now!

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

dataset = TensorDataset(X, y)

val_frac   = 0.2
val_size   = int(len(dataset) * val_frac)
train_size = len(dataset) - val_size
generator  = torch.Generator().manual_seed(42)
train_set, val_set = random_split(dataset, [train_size, val_size], generator=generator)

batch_size   = 64
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=batch_size)

### 25.2 Model Definition and Hyperparameters

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# --- Hyperparameters ---
n_embd    = 32    # embedding dimension (also the width of each block)
n_heads   = 4     # number of attention heads per block (head_size = n_embd // n_heads = 8)
n_blocks  = 2     # how many transformer blocks to stack
dropout   = 0.2   # dropout probability for regularisation

learning_rate = 0.001
epochs        = 20

model = TransformerDecoder(
    vocab_size   = vocab_size,
    sequence_len = sequence_len,
    n_embd       = n_embd,
    n_heads      = n_heads,
    n_blocks     = n_blocks,
    dropout      = dropout,
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal trainable parameters: {total_params:,}')
print(model)

In [ ]:
from torchview import draw_graph

dummy_input = torch.randint(0, vocab_size, (1, sequence_len), device=device)

with torch.no_grad():
    model_graph = draw_graph(model, input_data=dummy_input)
model_graph.visual_graph

---

🖍 **Exercise 25.1:** The transformer can in principle compute attention between *any* two positions. But during training we apply a causal mask that prevents each position from attending to future positions.

1. **Why** do we need this mask? What would go wrong if we trained without it?
2. The mask is a lower-triangular matrix of ones. Concretely, after masking, which positions can position $t = 3$ attend to in a sequence of length $T = 6$?

---

<!-- BEGIN QUESTION -->

*Your answer here.*

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---

🖍 **Exercise 25.2:** The LSTM processes tokens **sequentially** (one after another), while the transformer processes all tokens **in parallel**.

1. Why does the transformer need explicit **positional encoding** while the LSTM does not?
2. Consider two sequences: ``[C4, G4, E4]`` and ``[G4, E4, C4]``. Without positional encoding, would the transformer treat these the same or differently? Explain briefly.

---

<!-- BEGIN QUESTION -->

*Your answer here.*

_Type your answer here, replacing this text._

<!-- END QUESTION -->

### 25.3 Model Training

The training loop is similar to the LSTM, with one key difference: the model output has shape `(B, T, vocab_size)` instead of `(B, vocab_size)`. We flatten the batch and time dimensions before computing the cross-entropy loss so that we train on all `T` positions at once.

In [ ]:
from tqdm.auto import tqdm
import seaborn as sns
import matplotlib.pyplot as plt

loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

train_losses, val_losses = [], []
train_accs,   val_accs   = [], []

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    context = torch.enable_grad() if train else torch.no_grad()

    desc = f"Epoch {epoch}/{epochs} {'train' if train else 'val  '}"
    pbar = tqdm(loader, desc=desc, leave=False)

    with context:
        for xb, yb in pbar:
            xb, yb = xb.to(device), yb.to(device)   # (B, T), (B, T)

            logits = model(xb)                        # (B, T, vocab_size)
            B, T, C = logits.shape

            # Flatten time and batch for loss computation
            loss = loss_fn(logits.view(B * T, C), yb.view(B * T))

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * (B * T)
            correct    += (logits.argmax(dim=-1) == yb).sum().item()
            total      += B * T

            pbar.set_postfix(loss=f"{total_loss/total:.4f}", acc=f"{correct/total:.4f}")

    return total_loss / total, correct / total

for epoch in range(1, epochs + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader,   train=False)

    train_losses.append(tr_loss); train_accs.append(tr_acc)
    val_losses.append(va_loss);   val_accs.append(va_acc)

    print(f"Epoch {epoch:3d}/{epochs}  "
          f"train  loss {tr_loss:.4f}  acc {tr_acc:.4f}   |   "
          f"val  loss {va_loss:.4f}  acc {va_acc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.lineplot(x=range(len(train_losses)), y=train_losses, linewidth=2.5, label='train', ax=axes[0])
sns.lineplot(x=range(len(val_losses)),   y=val_losses,   linewidth=2.5, label='val',   ax=axes[0])
axes[0].set_title("Cross-entropy loss")
axes[0].set_xlabel("Epochs")
axes[0].set_ylabel("Loss")

sns.lineplot(x=range(len(train_accs)), y=train_accs, linewidth=2.5, label='train', ax=axes[1])
sns.lineplot(x=range(len(val_accs)),   y=val_accs,   linewidth=2.5, label='val',   ax=axes[1])
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epochs")
axes[1].set_ylabel("Accuracy")

plt.tight_layout()
plt.show()

---

🖍 **Exercise 25.3:** Count the learnable parameters of our transformer decoder.

Work through each component and compute the number of parameters. Then verify your result using `sum(p.numel() for p in model.parameters() if p.requires_grad)`.

**Components to consider** (with the default settings `vocab_size=30, sequence_len=64, n_embd=32, n_heads=4, head_size=8, n_blocks=2`):

| Component | Formula | Value |
|-----------|---------|-------|
| `token_embedding_table` | `vocab_size × n_embd` | ? |
| `position_embedding_table` | `sequence_len × n_embd` | ? |
| `key` (per head, no bias) | `n_embd × head_size` | ? |
| `query` (per head, no bias) | `n_embd × head_size` | ? |
| `value` (per head, no bias) | `n_embd × head_size` | ? |
| `W0` (with bias) | `(head_size × n_heads) × n_embd + n_embd` | ? |
| `FeedForward` linear 1 (with bias) | `n_embd × (4·n_embd) + 4·n_embd` | ? |
| `FeedForward` linear 2 (with bias) | `(4·n_embd) × n_embd + n_embd` | ? |
| `LayerNorm` (scale + bias) | `2 × n_embd` | ? |
| `lm_head` (with bias) | `n_embd × vocab_size + vocab_size` | ? |
| `ln_f` (final LayerNorm) | `2 × n_embd` | ? |

---

In [ ]:
vocab_size_   = vocab_size
sequence_len_ = sequence_len
n_embd_       = n_embd
n_heads_      = n_heads
head_size_    = n_embd_ // n_heads_
n_blocks_     = n_blocks

token_emb_params    = ...
pos_emb_params      = ...

# Parameters per attention head (key + query + value, no bias)
params_per_head     = ...

# Parameters for W0 (n_heads * head_size -> n_embd, with bias)
W0_params           = ...

# Parameters for one FeedForward block
ffn_params          = ...

# Parameters for two LayerNorm layers inside one Block
layernorm_params    = ...

# Total parameters per Block
params_per_block    = n_heads_ * params_per_head + W0_params + ffn_params + layernorm_params

# Final layer norm + output head
ln_f_params         = ...
lm_head_params      = ...

total = token_emb_params + pos_emb_params + n_blocks_ * params_per_block + ln_f_params + lm_head_params
print(f'Calculated: {total}')
print(f'Actual:     {sum(p.numel() for p in model.parameters() if p.requires_grad)}')

In [ ]:
grader.check("q253")

### 25.4 Save your Model (optional)

In [ ]:
import os
os.makedirs('models', exist_ok=True)
torch.save(model.state_dict(), 'models/transformer_model.pt')

### 25.5 Load a Model (optional)

In [ ]:
model = TransformerDecoder(
    vocab_size   = vocab_size,
    sequence_len = sequence_len,
    n_embd       = n_embd,
    n_heads      = n_heads,
    n_blocks     = n_blocks,
    dropout      = dropout,
).to(device)

model.load_state_dict(torch.load('models/transformer_model.pt', map_location=device))
model.eval()

### 25.6 Generating Melodies

Generation works exactly like for the LSTM: we start with a seed melody, repeatedly ask the model to predict the next token, append it to the growing melody, and stop when the model predicts the ``TERM_SYMBOL`` or we reach a maximum length.

The only difference is that the transformer's `forward` returns shape `(B, T, vocab_size)`, so we take the **last position's logit** — `logits[0, -1, :]` — as the prediction for the next token.

In [ ]:
def generate_melody(
    model: TransformerDecoder,
    sequence_len: int,
    seed: list,
    string_to_int: StringToIntEncoder,
    temperature: float = 1.0,
    max_len: int = 1_000,
) -> list:
    """Auto-regressively generate a melody token by token."""

    # Pad seed so the model knows we are at the beginning of a melody
    padded_seed = [TERM_SYMBOL] * sequence_len + seed
    seed_int    = string_to_int.encode_sequence(padded_seed)

    melody = seed[:]
    model.eval()
    with torch.no_grad():
        while True:
            # Always feed the most recent `sequence_len` tokens
            window = seed_int[-sequence_len:]
            idx    = torch.tensor([window], dtype=torch.long, device=device)

            logits      = model(idx)          # (1, T, vocab_size)
            logits_last = logits[0, -1, :]    # (vocab_size,) — prediction for the next token

            if temperature <= 0:
                symbol_int = logits_last.argmax(dim=-1).item()
            else:
                probs      = torch.softmax(logits_last / temperature, dim=-1)
                symbol_int = torch.multinomial(probs, num_samples=1).item()

            seed_int.append(symbol_int)
            symbol = string_to_int.decode(symbol_int)

            if symbol == TERM_SYMBOL:
                break
            melody.append(symbol)
            if len(melody) >= max_len:
                break

    return melody

In [ ]:
seed = ['60', '_', '_', '55', '_', '_']

melody = generate_melody(
    model        = model,
    sequence_len = sequence_len,
    seed         = seed,
    string_to_int= string_to_int,
    temperature  = 0.8,
    max_len      = 100,
)
print(melody)

score = piano_roll_encoder.decode_stream(melody)
score.show('midi')

---

🖍 **Exercise 25.4:** Experiment with the ``temperature`` parameter in ``generate_melody``.

1. Try at least three different values (e.g. `0.2`, `1.0`, `2.0`). Listen to the generated melodies. How does the musical character change?
2. What happens in the limit as `temperature → 0`? What happens as `temperature → ∞`?
3. There is no single "correct" temperature. What considerations might guide your choice as a composer who wants to use this tool creatively?

---

<!-- BEGIN QUESTION -->

*Your answer here.*

_Type your answer here, replacing this text._

<!-- END QUESTION -->

### 25.7 Transformer vs LSTM: A Comparison

Now that you have trained both an LSTM and a transformer on the same data, it is worth comparing the two approaches:

| Property | LSTM | Transformer |
|----------|------|-------------|
| Processes sequence | Sequentially (token by token) | In parallel (all tokens at once) |
| Long-range dependencies | Via hidden state (can be washed out) | Via direct attention (all positions) |
| Memory during training | $\mathcal{O}(T)$ | $\mathcal{O}(T^2)$ |
| Training speed (GPU) | Slower (sequential) | Faster (parallel) |
| Suitable for very long sequences | Yes (bounded memory) | Challenging (quadratic cost) |
| Training signals per window | 1 | T |

For short sequences (as in our folk song dataset) the differences in output quality are often small. The transformer tends to shine when sequences are longer and when very long-range context matters — which in music would correspond to learning larger-scale structure such as phrases, sections, or motific development.

---

🖍 **Exercise 25.5 (optional, open-ended):** Increase the capacity of the transformer by tuning the hyperparameters (e.g. `n_embd=64, n_heads=4, n_blocks=4`) and train for more epochs.

- Does the validation loss improve?
- Does the generated music sound better to you?
- At what point does the model start to overfit? How can you tell?

You could also try comparing the transformer and LSTM by training both with roughly the same number of parameters and evaluating the validation loss after the same number of epochs.

---

If you are interested in the topic, I encourage you to look into [Music Transformer: Generating Music with Long-Term Structure](https://magenta.withgoogle.com/music-transformer) and the blog series [Musical Interrogation](https://bzoennchen.github.io/Pages/2023/04/02/musical-interrogation-I.html).